# Extended SQL Practice: GROUP BY, Aggregations, HAVING & CASE

This notebook expands the optional GROUP BY practice with additional real-world style exercises on the **Chinook** database.

**Focus areas:**
- Basic and multi-column `GROUP BY`
- Aggregations: `COUNT`, `SUM`, `AVG`, `MIN`, `MAX`
- Filtering groups with `HAVING`
- Conditional logic with `CASE`
- Combining joins + aggregations (very common in interviews and day-to-day work)
- Ranking patterns and business questions

All queries target the Chinook schema shown below. Solutions are provided at the end of each major section so you can attempt the exercises first.

## Data Schema

![Database Schema](data-schema.png)

**Key tables you’ll use most:**
- `tracks` (TrackId, Name, AlbumId, MediaTypeId, GenreId, UnitPrice, Milliseconds, Bytes)
- `albums` (AlbumId, Title, ArtistId)
- `artists` (ArtistId, Name)
- `genres` (GenreId, Name)
- `invoices` (InvoiceId, CustomerId, InvoiceDate, BillingCountry, Total)
- `invoice_items` (InvoiceLineId, InvoiceId, TrackId, UnitPrice, Quantity)
- `customers` (CustomerId, FirstName, LastName, Country, SupportRepId, ...)
- `employees` (EmployeeId, FirstName, LastName, Title, ReportsTo, ...)
- `playlist_track` / `playlists`

---
## Section 1 – Foundations (Original Exercises)

### Exercise 1: Minimum and maximum track prices per album

The sales department wants to know the minimum and maximum track prices for each album.

**Instructions**
- Group tracks by `AlbumId`
- Return `AlbumId`, lowest track price (`MIN`), highest track price (`MAX`)
- Alias the aggregates as `LowestTrackPrice` and `HighestTrackPrice`

**Expected output (first 25 rows):**

```
+---------+------------------+-------------------+
| AlbumId | LowestTrackPrice | HighestTrackPrice |
+---------+------------------+-------------------+
| 1       | 0.99             | 0.99              |
| 2       | 0.99             | 0.99              |
...
(347 total rows)
```

### Exercise 2: Spending by invoice categories

Categorize invoices by total amount:
- **Low**   → Total < 2
- **Medium** → Total BETWEEN 2 AND 5
- **High**  → Total > 5
- **NA**    → anything else (missing)

Return the category and the number of invoices in each category.

**Expected output:**

```
+------------------+-------------+
| SpendingCategory | NumInvoices |
+------------------+-------------+
| High             | 179         |
| Low              | 170         |
| Medium           | 63          |
+------------------+-------------+
```

---
## Section 2 – Intermediate Aggregations

### Exercise 3: Tracks per genre with average length

Business question: *Which genres have the most tracks, and how long are those tracks on average?*

**Requirements**
- Join `tracks` and `genres`
- Group by genre name
- Return:
  - `Genre`
  - `TrackCount`
  - `AvgMilliseconds` (rounded to whole number)
  - `AvgMinutes` (AvgMilliseconds / 60000, rounded to 2 decimals)
- Order by `TrackCount` descending

**Hints**
- Use `ROUND(AVG(...), 0)` and `ROUND(AVG(...)/60000.0, 2)`
- Remember that SQLite division of integers truncates; force floating-point with `.0`

### Exercise 4: Revenue by country (top markets)

Calculate total revenue and number of invoices per billing country.

**Requirements**
- Use the `invoices` table
- Return `BillingCountry`, `TotalRevenue`, `InvoiceCount`
- Order by `TotalRevenue` descending
- Show only the top 10 countries

### Exercise 5: Artists with the most albums

Find how many albums each artist has released.

**Requirements**
- Join `artists` and `albums`
- Group by artist name
- Return `Artist`, `AlbumCount`
- Keep only artists who have **more than 5 albums** (`HAVING`)
- Order by `AlbumCount` descending

---
## Section 3 – Multi-table + Aggregations (Interview Style)

### Exercise 6: Total sales per customer

For each customer calculate:
- Full name (`FirstName || ' ' || LastName`)
- Country
- Number of invoices
- Total amount spent

Order by total spent descending and show the top 15 customers.

### Exercise 7: Best-selling tracks (by quantity sold)

Using `invoice_items` + `tracks`:

- Return track name, total quantity sold, and total revenue generated by that track
- Order by quantity sold descending
- Limit to the top 20 tracks

### Exercise 8: Genre performance by revenue

Which genres generate the most revenue?

**Path:** `invoice_items` → `tracks` → `genres`

Return:
- Genre name
- Number of distinct tracks sold from that genre
- Total units sold
- Total revenue

Order by total revenue descending.

### Exercise 9: Support representatives performance

Each customer has a `SupportRepId` that points to an employee.

Calculate for every support agent:
- Agent full name
- Number of customers they support
- Total revenue generated by their customers

Order by revenue descending.

---
## Section 4 – Advanced Patterns

### Exercise 10: Monthly revenue trend

Extract year and month from `InvoiceDate` and calculate total revenue per month.

**Requirements**
- Use `strftime('%Y-%m', InvoiceDate)` (or equivalent)
- Return `YearMonth`, `Revenue`, `InvoiceCount`
- Order chronologically

### Exercise 11: Customers who spent more than the average

Find customers whose total spend is **greater than the overall average customer spend**.

**Requirements**
- Calculate total spend per customer
- Compare against a subquery that returns the average of those totals
- Return customer name, country, and total spend
- Order by total spend descending

### Exercise 12: Album completeness check

For each album report:
- Album title
- Artist name
- Number of tracks
- Average track length in minutes
- Total album length in minutes

Keep only albums that have **at least 10 tracks** and order by total length descending.

### Exercise 13: Price tier analysis on tracks

Create price tiers for tracks:
- `Budget`   → UnitPrice < 1.00
- `Standard` → UnitPrice = 0.99 or 1.99 (most common)
- `Premium`  → UnitPrice > 1.99

Then count how many tracks fall into each tier and what percentage of all tracks they represent.

### Exercise 14: Playlist popularity

Which playlists contain the most tracks?

Join `playlists` and `playlist_track`, group by playlist name, return playlist name + track count, order by track count descending, show top 10.

---
## Section 5 – Challenge Questions (combine everything)

### Challenge 1: High-value customers in each country

For every country, find the single customer who spent the most.

Return: Country, Customer Name, Total Spent.

*(Hint: this usually requires a subquery or window function approach. With pure GROUP BY you can first find the max spend per country, then join back.)*

### Challenge 2: Genre concentration per artist

Some artists appear in multiple genres. Write a query that shows:

- Artist name
- Number of distinct genres they appear in
- List of those genres (use `GROUP_CONCAT`)

Keep only artists present in **2 or more genres** and order by number of genres descending.

### Challenge 3: Invoice size distribution with percentiles approximation

Using the earlier Low / Medium / High categories, also calculate the percentage of total invoices each category represents and the percentage of total revenue each category contributes.

---
# SOLUTIONS

Attempt the exercises above before looking here.

## Solution – Exercise 1

In [ ]:
SELECT AlbumId, 
       MIN(UnitPrice) AS LowestTrackPrice, 
       MAX(UnitPrice) AS HighestTrackPrice 
FROM tracks 
GROUP BY AlbumId;

## Solution – Exercise 2

In [ ]:
SELECT 
    CASE 
        WHEN Total < 2 THEN 'Low'
        WHEN Total BETWEEN 2 AND 5 THEN 'Medium'
        WHEN Total > 5 THEN 'High'
        ELSE 'NA'
    END AS SpendingCategory,
    COUNT(*) AS NumInvoices
FROM invoices
GROUP BY SpendingCategory;

## Solution – Exercise 3

In [ ]:
SELECT 
    g.Name AS Genre,
    COUNT(*) AS TrackCount,
    ROUND(AVG(t.Milliseconds), 0) AS AvgMilliseconds,
    ROUND(AVG(t.Milliseconds) / 60000.0, 2) AS AvgMinutes
FROM tracks t
JOIN genres g ON t.GenreId = g.GenreId
GROUP BY g.Name
ORDER BY TrackCount DESC;

## Solution – Exercise 4

In [ ]:
SELECT 
    BillingCountry,
    ROUND(SUM(Total), 2) AS TotalRevenue,
    COUNT(*) AS InvoiceCount
FROM invoices
GROUP BY BillingCountry
ORDER BY TotalRevenue DESC
LIMIT 10;

## Solution – Exercise 5

In [ ]:
SELECT 
    ar.Name AS Artist,
    COUNT(*) AS AlbumCount
FROM artists ar
JOIN albums al ON ar.ArtistId = al.ArtistId
GROUP BY ar.Name
HAVING COUNT(*) > 5
ORDER BY AlbumCount DESC;

## Solution – Exercise 6

In [ ]:
SELECT 
    c.FirstName || ' ' || c.LastName AS CustomerName,
    c.Country,
    COUNT(i.InvoiceId) AS InvoiceCount,
    ROUND(SUM(i.Total), 2) AS TotalSpent
FROM customers c
JOIN invoices i ON c.CustomerId = i.CustomerId
GROUP BY c.CustomerId, c.FirstName, c.LastName, c.Country
ORDER BY TotalSpent DESC
LIMIT 15;

## Solution – Exercise 7

In [ ]:
SELECT 
    t.Name AS TrackName,
    SUM(ii.Quantity) AS UnitsSold,
    ROUND(SUM(ii.UnitPrice * ii.Quantity), 2) AS Revenue
FROM invoice_items ii
JOIN tracks t ON ii.TrackId = t.TrackId
GROUP BY t.TrackId, t.Name
ORDER BY UnitsSold DESC
LIMIT 20;

## Solution – Exercise 8

In [ ]:
SELECT 
    g.Name AS Genre,
    COUNT(DISTINCT t.TrackId) AS DistinctTracksSold,
    SUM(ii.Quantity) AS UnitsSold,
    ROUND(SUM(ii.UnitPrice * ii.Quantity), 2) AS TotalRevenue
FROM invoice_items ii
JOIN tracks t ON ii.TrackId = t.TrackId
JOIN genres g ON t.GenreId = g.GenreId
GROUP BY g.Name
ORDER BY TotalRevenue DESC;

## Solution – Exercise 9

In [ ]:
SELECT 
    e.FirstName || ' ' || e.LastName AS SupportAgent,
    COUNT(DISTINCT c.CustomerId) AS CustomersSupported,
    ROUND(SUM(i.Total), 2) AS TotalRevenue
FROM employees e
JOIN customers c ON e.EmployeeId = c.SupportRepId
JOIN invoices i ON c.CustomerId = i.CustomerId
GROUP BY e.EmployeeId, e.FirstName, e.LastName
ORDER BY TotalRevenue DESC;

## Solution – Exercise 10

In [ ]:
SELECT 
    strftime('%Y-%m', InvoiceDate) AS YearMonth,
    ROUND(SUM(Total), 2) AS Revenue,
    COUNT(*) AS InvoiceCount
FROM invoices
GROUP BY strftime('%Y-%m', InvoiceDate)
ORDER BY YearMonth;

## Solution – Exercise 11

In [ ]:
WITH customer_totals AS (
    SELECT 
        c.CustomerId,
        c.FirstName || ' ' || c.LastName AS CustomerName,
        c.Country,
        SUM(i.Total) AS TotalSpent
    FROM customers c
    JOIN invoices i ON c.CustomerId = i.CustomerId
    GROUP BY c.CustomerId, c.FirstName, c.LastName, c.Country
)
SELECT CustomerName, Country, ROUND(TotalSpent, 2) AS TotalSpent
FROM customer_totals
WHERE TotalSpent > (SELECT AVG(TotalSpent) FROM customer_totals)
ORDER BY TotalSpent DESC;

## Solution – Exercise 12

In [ ]:
SELECT 
    al.Title AS Album,
    ar.Name AS Artist,
    COUNT(*) AS TrackCount,
    ROUND(AVG(t.Milliseconds) / 60000.0, 2) AS AvgMinutes,
    ROUND(SUM(t.Milliseconds) / 60000.0, 2) AS TotalMinutes
FROM albums al
JOIN artists ar ON al.ArtistId = ar.ArtistId
JOIN tracks t ON al.AlbumId = t.AlbumId
GROUP BY al.AlbumId, al.Title, ar.Name
HAVING COUNT(*) >= 10
ORDER BY TotalMinutes DESC;

## Solution – Exercise 13

In [ ]:
SELECT 
    CASE 
        WHEN UnitPrice < 1.00 THEN 'Budget'
        WHEN UnitPrice IN (0.99, 1.99) THEN 'Standard'
        WHEN UnitPrice > 1.99 THEN 'Premium'
        ELSE 'Other'
    END AS PriceTier,
    COUNT(*) AS TrackCount,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM tracks), 2) AS Percentage
FROM tracks
GROUP BY PriceTier
ORDER BY TrackCount DESC;

## Solution – Exercise 14

In [ ]:
SELECT 
    p.Name AS Playlist,
    COUNT(*) AS TrackCount
FROM playlists p
JOIN playlist_track pt ON p.PlaylistId = pt.PlaylistId
GROUP BY p.PlaylistId, p.Name
ORDER BY TrackCount DESC
LIMIT 10;

## Solution – Challenge 1 (High-value customer per country)

In [ ]:
WITH customer_spend AS (
    SELECT 
        c.CustomerId,
        c.FirstName || ' ' || c.LastName AS CustomerName,
        c.Country,
        SUM(i.Total) AS TotalSpent
    FROM customers c
    JOIN invoices i ON c.CustomerId = i.CustomerId
    GROUP BY c.CustomerId, c.FirstName, c.LastName, c.Country
),
max_per_country AS (
    SELECT Country, MAX(TotalSpent) AS MaxSpent
    FROM customer_spend
    GROUP BY Country
)
SELECT 
    cs.Country,
    cs.CustomerName,
    ROUND(cs.TotalSpent, 2) AS TotalSpent
FROM customer_spend cs
JOIN max_per_country m 
  ON cs.Country = m.Country AND cs.TotalSpent = m.MaxSpent
ORDER BY cs.TotalSpent DESC;

## Solution – Challenge 2 (Genre concentration per artist)

In [ ]:
SELECT 
    ar.Name AS Artist,
    COUNT(DISTINCT g.GenreId) AS GenreCount,
    GROUP_CONCAT(DISTINCT g.Name) AS Genres
FROM artists ar
JOIN albums al ON ar.ArtistId = al.ArtistId
JOIN tracks t ON al.AlbumId = t.AlbumId
JOIN genres g ON t.GenreId = g.GenreId
GROUP BY ar.ArtistId, ar.Name
HAVING COUNT(DISTINCT g.GenreId) >= 2
ORDER BY GenreCount DESC, Artist;

## Solution – Challenge 3 (Invoice size distribution + revenue share)

In [ ]:
WITH categorized AS (
    SELECT 
        CASE 
            WHEN Total < 2 THEN 'Low'
            WHEN Total BETWEEN 2 AND 5 THEN 'Medium'
            WHEN Total > 5 THEN 'High'
            ELSE 'NA'
        END AS SpendingCategory,
        Total
    FROM invoices
)
SELECT 
    SpendingCategory,
    COUNT(*) AS NumInvoices,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM categorized), 2) AS PctOfInvoices,
    ROUND(SUM(Total), 2) AS TotalRevenue,
    ROUND(100.0 * SUM(Total) / (SELECT SUM(Total) FROM categorized), 2) AS PctOfRevenue
FROM categorized
GROUP BY SpendingCategory
ORDER BY TotalRevenue DESC;

---
## Quick Reference – Patterns Used in This Notebook

| Pattern | When to use |
|---------|-------------|
| `GROUP BY col` | Aggregate per category |
| `HAVING condition` | Filter **after** aggregation |
| `CASE WHEN ... END` | Create buckets / labels |
| `COUNT(DISTINCT ...)` | Unique items inside a group |
| `GROUP_CONCAT` | List values inside a group (SQLite) |
| CTEs (`WITH ...`) | Break complex logic into readable steps |
| Subquery in `WHERE` / `HAVING` | Compare against overall averages or maxima |
| `strftime` | Extract year/month from dates |

Practice writing the queries from scratch first. Once comfortable, try modifying the business questions (e.g., “only 2012 data”, “exclude one genre”, “top 3 per country”). That is the fastest way to become job-ready with SQL.